# Folio-OCR on free GPUs: Phase 0 checks and the Phase 1 resume model

Runs on **Kaggle** (Settings: *Accelerator* = **GPU T4 x2**, *Internet* = **On**, which needs a phone-verified
account) and also on **free Colab** (one T4; Runtime > Change runtime type > T4 GPU).

What it does, in order:

1. installs the code and its assets (fonts, KaTeX);
2. gets the synthetic data (rendered here, or uploaded after rendering it on your own computer, which saves GPU hours);
3. **Phase 0**: shows Infinity-Parser2-Flash's raw output on one page, scores the base model on the dev pages,
   and (optional) on a 300-page sample of olmOCR-Bench;
4. **Phase 1**: fine-tunes Flash with 4-bit LoRA on resumes + general pages, then scores the result against the base model.

Every long step can be re-run after a session ends: rendering appends, benchmarks skip finished pages, training resumes from the last checkpoint.

In [ ]:
import os, glob, shutil, subprocess, sys, zipfile
ON_KAGGLE = os.path.exists("/kaggle")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
CODE, DATA, RUNS = f"{WORK}/Folio-OCR", f"{WORK}/data", f"{WORK}/runs"
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "no GPU visible")
N_GPUS = int(subprocess.run("nvidia-smi -L | wc -l", shell=True, capture_output=True, text=True).stdout.strip() or 0)
print("GPUs:", N_GPUS)

## 1. Code and dependencies

Upload `Folio-OCR.zip` as a Kaggle dataset (Datasets > New dataset; Kaggle unpacks it) and add it to this notebook,
or push the folder to your GitHub and set `GIT_URL`.

In [ ]:
GIT_URL = ""   # e.g. "https://github.com/<you>/Folio-OCR"
if not os.path.exists(f"{CODE}/folioocr"):
    found = glob.glob("/kaggle/input/**/folioocr/__init__.py", recursive=True)
    zips = glob.glob("/kaggle/input/**/Folio-OCR*.zip", recursive=True) + glob.glob("/content/Folio-OCR*.zip")
    if found:
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), CODE)
    elif zips:
        zipfile.ZipFile(zips[0]).extractall(WORK)
    elif GIT_URL:
        subprocess.run(["git", "clone", GIT_URL, CODE], check=True)
    else:
        raise SystemExit("add the Folio-OCR dataset/zip or set GIT_URL")
os.chdir(CODE)
print(os.getcwd())

In [ ]:
!pip install -q "transformers==5.21.0" "peft==0.21.2" "accelerate>=1.10" "bitsandbytes>=0.45" \
    pypdfium2 faker fonttools rapidfuzz fuzzysearch beautifulsoup4 pypdf playwright apted
!pip install -q --no-deps "olmocr==0.4.27"
!pip uninstall -y -q torchao   # Kaggle's torchao is too old for peft
!python -m playwright install --with-deps chromium > /dev/null 2>&1 && echo chromium ok
!python scripts/fetch_assets.py
!CUDA_VISIBLE_DEVICES= python -m pytest -q tests 2>&1 | tail -3

## 2. Data

Rendering is CPU work, so the cheapest place to do it is your own computer
(`python -m folioocr.render.make_dataset ...`, see the README), then upload `data/` as a Kaggle dataset and set
`DATA_FROM_INPUT`. Rendering here takes about 40-60 minutes of the session for the sizes below.

In [ ]:
DATA_FROM_INPUT = ""      # e.g. "/kaggle/input/folio-data/data" if you uploaded rendered data
N_RESUMES, N_GENERAL = 5000, 2500
if DATA_FROM_INPUT:
    DATA = DATA_FROM_INPUT
else:
    workers = os.cpu_count() or 2
    !python -m folioocr.render.make_dataset --out {DATA}/resumes --n {N_RESUMES} --workers {workers} --mix "resume=1" --prefix r
    !python -m folioocr.render.make_dataset --out {DATA}/general --n {N_GENERAL} --workers {workers} --prefix g --start-seed 1000000 \
        --mix "article=0.35,table=0.3,math=0.15,letter=0.1,indic=0.1"
!wc -l {DATA}/resumes/labels.jsonl {DATA}/general/labels.jsonl

## 3. Phase 0

**3a. What does the base model write?** Infinity-Parser2-Flash's `doc2json` output format is not documented beyond its
post-processor. Look at one raw output: if it is **not** wrapped in a ```` ```json ```` fence, add `--no-fence` to the
training command in step 4 so the targets match the model's own habit.

In [ ]:
dev_img = sorted(glob.glob(f"{DATA}/resumes/images/*.jpg"))[0]
!python scripts/infer.py {dev_img} --raw --max-new-tokens 3000 --out {WORK}/phase0_raw | head -40

**3b. Base model on the dev pages** (about 3% of the rendered pages, held out from training).

In [ ]:
!python scripts/bench_dev.py --labels {DATA}/resumes/labels.jsonl --tests {DATA}/resumes/tests.jsonl --name flash_base \
    --batch-size 4 --max-new-tokens 4096 --limit 60

**3c. (Optional, about 1-2 hours) olmOCR-Bench, 300-page stratified sample, common 1,288 px rendering.**
Run the same command with `--engine chandra` / `--engine surya` (after `pip install chandra-ocr[hf] surya-ocr`) for the
other baselines; the same `--seed` gives every engine the same pages. Gate G0 compares these with the published scores.

In [ ]:
RUN_OLMOCR = False
if RUN_OLMOCR:
    from huggingface_hub import snapshot_download
    snapshot_download("allenai/olmOCR-bench", repo_type="dataset", local_dir=f"{WORK}/olmOCR-bench")
    !python scripts/bench_olmocr.py --bench {WORK}/olmOCR-bench/bench_data --name flash_common_s300 --sample 300 \
        --batch-size 4 --max-new-tokens 6000 --cache {WORK}/bench_cache

## 4. Phase 1: the resume model (4-bit LoRA on Flash)

Measure first: `MAX_STEPS = 20` shows seconds per step on this hardware; then set it to `-1` for the full epoch.
With two T4s the run uses both (DDP); dev evaluation then happens once, at the end.
If the session ends, run this cell again: `--resume` continues from the last checkpoint.

In [ ]:
MAX_STEPS = 20
FENCE_FLAG = ""          # "--no-fence" if step 3a showed un-fenced JSON
launcher = f"accelerate launch --num_processes {N_GPUS} --mixed_precision fp16" if N_GPUS > 1 else "python"
!{launcher} scripts/train_sft.py --profile t4 \
    --data {DATA}/resumes/labels.jsonl:3 {DATA}/general/labels.jsonl:2 \
    --dev {DATA}/resumes/labels.jsonl --dev-tests {DATA}/resumes/tests.jsonl \
    --out {RUNS}/resume-t4 --epochs 1 --max-steps {MAX_STEPS} --eval-every 300 --save-every 150 --eval-pages 12 \
    --resume {FENCE_FLAG}

## 5. Did it help? Same dev pages, paired bootstrap

`scripts/compare.py` prints the score difference with its 95% interval; the plan only counts a win when that interval
excludes zero. For the gate G1 claim, score both models on **Resume-Bench** (real resumes, `scripts/bench_resume.py`) as well.

In [ ]:
!python scripts/bench_dev.py --labels {DATA}/resumes/labels.jsonl --tests {DATA}/resumes/tests.jsonl --name resume_v1 \
    --adapter {RUNS}/resume-t4/adapter --batch-size 4 --max-new-tokens 4096 --limit 60
!python scripts/compare.py results/dev/resume_v1.json results/dev/flash_base.json

## 6. Keep the results

On Kaggle, everything under `/kaggle/working` is saved with the notebook version (*Save Version*): the adapter in
`runs/resume-t4/adapter` is about 100-200 MB. On Colab, copy it to Google Drive. To share it:
`hf upload <you>/folio-ocr-resume-v1 runs/resume-t4/adapter`.

In [ ]:
!du -sh {RUNS}/resume-t4/adapter results 2>/dev/null
!ls {RUNS}/resume-t4